# Lab Exercise: AI Model Monitoring with Drift Detection
## AIAT 125 — Unit 5: Monitoring and Maintaining Deployed AI Models

**Learning objectives**
1. Record a baseline model performance snapshot (accuracy, precision, recall, F1, latency).
2. Simulate production data drift (data drift + concept drift) realistically.
3. Apply the Kolmogorov-Smirnov test to detect feature distribution shift.
4. Build a `MonitoringSystem` class that encapsulates all checks and generates a structured report.

**Why monitoring matters**  
A model that scores 93% accuracy in testing can drop to 40% in production six months later — not because the code changed, but because the world changed. Monitoring is the only way to catch this before users notice.

**Grading** — 100 points total
| Task | Points |
|---|---|
| Task 1: Record baseline metrics | 20 |
| Task 2: Simulate and extend drift | 25 |
| Task 3: KS test across all features | 25 |
| Task 4: `MonitoringSystem` class | 30 |


## Setup — Run this cell first (do not modify)

This mirrors the official Tuwaiq Academy lab setup exactly. The dataset and model are fixed so your results match the expected output.

In [ ]:
# WHAT: train the baseline model on a fixed synthetic dataset (seeded — do not change).
# WHY: every task below measures drift and degradation AGAINST this baseline;
# a shared random_state means your numbers match the expected asserts.
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings("ignore")

# Fixed dataset — do not change random_state
X, y = make_classification(
    n_samples=5000,
    n_features=6,
    n_informative=4,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train baseline model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Features         : {X.shape[1]}")
print("Setup complete — model trained.")

---
## Task 1 — Record Baseline Metrics (20 points)

Measure the model's performance on the **test set** (the reference distribution). These numbers become your ground truth for detecting future degradation.

Five things to capture (four quality metrics plus latency):
- **Accuracy**: overall correctness
- **Precision**: when the model says positive, how often is it right?
- **Recall**: of all actual positives, how many did the model catch?
- **F1-score**: harmonic mean of precision and recall (use when classes are imbalanced)
- **Inference latency**: how long does `model.predict()` take?

**Expected output** (from the official lab):
```
=== Baseline Model Metrics ===
Accuracy : 0.9307
Precision: 0.9380
Recall   : 0.9231
F1-Score : 0.9305
Latency  : ~0.076 s
```

In [ ]:
# --- Task 1: Fill in the blanks ---

start_time = time.time()
baseline_predictions = model.predict(X_test)
baseline_latency = time.time() - start_time

# TODO: compute the four metrics using sklearn.metrics functions
# Replace None with the correct function calls
baseline_accuracy  = None   # accuracy_score(...)
baseline_precision = None   # precision_score(...)
baseline_recall    = None   # recall_score(...)
baseline_f1        = None   # f1_score(...)

# TODO: print a formatted baseline report (match the expected output above)
# YOUR CODE HERE

# Validation
assert baseline_accuracy  is not None, "Compute baseline_accuracy"
assert baseline_precision is not None, "Compute baseline_precision"
assert baseline_recall    is not None, "Compute baseline_recall"
assert baseline_f1        is not None, "Compute baseline_f1"
assert baseline_accuracy  > 0.90, f"Expected >90%, got {baseline_accuracy:.2%}"
print("Task 1 PASSED")

---
## Task 2 — Simulate Production Data Drift (25 points)

In production, two types of drift can occur:

| Drift type | What changes | Effect |
|---|---|---|
| **Data drift** | Input feature distributions shift | Same model, different-looking data |
| **Concept drift** | The relationship between X and y changes | Model logic becomes wrong |

The official lab introduces drift on **feature 0** (add 2). You will extend this to also shift **feature 2** (add 1.5), making drift more realistic.

**Why it matters:** A credit-scoring model trained on pre-pandemic data will see data drift when post-pandemic financial patterns emerge. If people's spending habits change (concept drift), the model's predictions become systematically wrong.

In [ ]:
# --- Task 2: Simulate production data with drift ---

# Start from a copy of the test data
X_production = X_test.copy()

# Official lab drift (provided — do not remove)
X_production[:, 0] = X_production[:, 0] + 2   # data drift on feature 0

# TODO: Add a second data drift on feature 2 (shift by +1.5)
# YOUR CODE HERE

# Concept drift: target relationship changes
# (the model was trained on different y; now y depends only on feature 1)
y_production = np.where(X_production[:, 1] > 0, 1, 0)

# TODO: Print the mean of feature 0 and feature 2 before and after drift
# to show they shifted. Format: "Feature 0: before={:.2f}, after={:.2f}"
# YOUR CODE HERE

# TODO: Add a comment (# ...) explaining in one sentence why concept drift
# causes accuracy to drop even when the model code hasn't changed.

# Validation
assert not np.array_equal(X_production, X_test), "X_production must differ from X_test"
assert X_production[:, 0].mean() > X_test[:, 0].mean(), "Feature 0 should have shifted up"
assert X_production[:, 2].mean() > X_test[:, 2].mean(), "Feature 2 should have shifted up"
print("Task 2 PASSED")

---
## Task 3 — KS Test Across All Features (25 points)

The **Kolmogorov-Smirnov two-sample test** (`ks_2samp`) compares two distributions. The p-value is the probability of a gap **at least this large** if both samples came from the same distribution (Course 03 (AIAT 113), Unit 5, lesson 07) — so `p < 0.05` is the threshold at which you *flag* a feature for investigation, not proof that it drifted.

The official lab only checks feature 0. You will check **all 6 features** and build a drift summary.

**Reference:** `ks_stat, p_value = ks_2samp(reference_data, production_data)`

In [ ]:
# --- Task 3: KS test across all features ---

n_features = X_test.shape[1]

# TODO: Loop through all features (0 to n_features-1).
# For each feature:
#   1. Run ks_2samp(X_test[:, i], X_production[:, i])
#   2. Store result in a list called drift_results
#      Each entry: {"feature": i, "ks_stat": ..., "p_value": ..., "drift_detected": p < 0.05}
#   3. Print: f"Feature {i}: KS={ks_stat:.4f}, p={p_value:.2e}, {'DRIFT' if drift else 'stable'}"

drift_results = []  # fill this in your loop

# YOUR CODE HERE

# Summary
drifted_features = [r["feature"] for r in drift_results if r["drift_detected"]]
print(f"\nFeatures with drift: {drifted_features} / {n_features} total")

# Validation
assert len(drift_results) == n_features, f"Expected {n_features} results, got {len(drift_results)}"
assert any(r["drift_detected"] for r in drift_results), "At least features 0 and 2 should show drift"
assert drift_results[0]["drift_detected"], "Feature 0 must show drift (shifted by +2)"
assert drift_results[2]["drift_detected"], "Feature 2 must show drift (shifted by +1.5)"
print("Task 3 PASSED")

---
## Task 4 — `MonitoringSystem` Class (30 points)

In production, you don't run ad-hoc scripts — you have a monitoring class that encapsulates all checks and generates a structured report. This is what MLOps platforms like MLflow, Evidently AI, and Arize do internally.

Implement the four methods below. The scaffold is provided; fill in each method body.

In [ ]:
# --- Task 4: MonitoringSystem class ---

class MonitoringSystem:
    """Encapsulates all production monitoring checks for a deployed model."""

    def __init__(self):
        self.baseline = {}
        self.production = {}
        self.drift_report = {}

    def record_baseline(self, model, X_test, y_test):
        """Measure and store baseline metrics from the test set.
        Must populate self.baseline with keys:
        accuracy, precision, recall, f1, latency_s
        """
        # TODO: implement this method
        # Hint: measure latency with time.time() around model.predict()
        # YOUR CODE HERE
        pass

    def check_drift(self, X_reference, X_production, threshold=0.05):
        """Run KS test on every feature. Store in self.drift_report.
        self.drift_report should be a dict: {feature_index: {ks_stat, p_value, drift_detected}}
        Return True if ANY feature shows drift.
        """
        # TODO: implement this method
        # YOUR CODE HERE
        pass

    def check_performance(self, model, X_production, y_production, threshold=0.1):
        """Compare production accuracy against baseline. Store in self.production.
        self.production must have keys: accuracy, latency_s, degraded (bool)
        Return True if performance degraded (accuracy dropped by more than threshold).
        """
        # TODO: implement this method
        # YOUR CODE HERE
        pass

    def generate_report(self):
        """Return a structured dict summarising all monitoring results.
        Required keys: baseline_accuracy, production_accuracy, accuracy_drop,
        data_drift_detected, drifted_features, performance_degraded
        """
        # TODO: implement this method
        # YOUR CODE HERE
        pass


# --- Run the monitoring system ---
monitor = MonitoringSystem()
monitor.record_baseline(model, X_test, y_test)
monitor.check_drift(X_test, X_production)
monitor.check_performance(model, X_production, y_production)
report = monitor.generate_report()

print("=== Monitoring Summary Report ===")
for key, value in report.items():
    print(f"  {key}: {value}")

In [ ]:
# WHAT: the final gate — asserts that your report dict contains every required
# key with values in the expected ranges.
# WHY: the checks encode the story your pipeline should have found: high
# baseline, collapsed production accuracy, drift in features 0 and 2.
# --- Final validation ---

assert isinstance(report, dict), "generate_report() must return a dict"

required_keys = ["baseline_accuracy", "production_accuracy", "accuracy_drop",
                 "data_drift_detected", "drifted_features", "performance_degraded"]
for k in required_keys:
    assert k in report, f"Report missing key: '{k}'"

assert report["baseline_accuracy"] > 0.90, "Baseline accuracy should be > 90%"
assert report["production_accuracy"] < 0.60, "Production accuracy should drop significantly"
assert report["data_drift_detected"] == True, "Drift should be detected"
assert report["performance_degraded"] == True, "Degradation should be detected"
assert len(report["drifted_features"]) >= 2, "At least features 0 and 2 should drift"

# Reaching this line means every monitoring task detected what it should.
print("=== ALL TASKS PASSED ===")
print(f"Baseline accuracy    : {report['baseline_accuracy']:.2%}")
print(f"Production accuracy  : {report['production_accuracy']:.2%}")
print(f"Accuracy drop        : {report['accuracy_drop']:.2%}")
print(f"Data drift detected  : {report['data_drift_detected']}")
print(f"Drifted features     : {report['drifted_features']}")
print(f"Performance degraded : {report['performance_degraded']}")

---
## Self-Check Questions

1. **Why is the KS test useful for drift detection?** What does a p-value of 0.0001 tell you about feature 0's distribution shift?
2. **What is the difference between data drift and concept drift?** Give a real-world example of each (not the iris/classification examples used here).
3. **Why is monitoring p95 latency more useful than average latency?** What kind of user experience problem does high p95 latency cause?
4. **If baseline accuracy is 93% and production accuracy is 89% (a 4% drop), should you trigger a retraining alert if your threshold is 10%?** Justify your answer.

---
## Key KPI Categories (from curriculum)

| KPI Category | Examples | What it catches |
|---|---|---|
| Predictive | Accuracy, Precision, Recall, F1 | Model correctness degrading |
| Operational | Latency, Throughput, CPU/GPU | Infrastructure bottlenecks |
| Data Quality | KS statistic, missing values, PSI | Input distribution shifts |
| Business | Revenue impact, fraud loss reduction | Whether AI delivers value |
| Ethical/Fairness | Demographic parity, disparate impact | Bias creeping into production |